# AI Subscription Manager — Agent Demo

This demo proves the three required course outcomes:

- **Tools:** the agent calls real subscription tools.
- **Multi-step behavior:** it uses tool results to decide what to do next.
- **Memory:** information added earlier is available in a later request.

Run the cells from top to bottom using the project's `venv` kernel.

In [1]:
from pathlib import Path
import sys

# Find the project folder.
PROJECT_ROOT = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "src" / "agent.py").exists() and (p / "src" / "database.py").exists():
        PROJECT_ROOT = p
        break

# Fallback for the current local project.
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\CODEPLAY\subscription-manager-agent")

if not (PROJECT_ROOT / "src" / "agent.py").exists():
    raise RuntimeError("Open this notebook from your subscription-manager-agent project.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import database
database.DB_PATH = PROJECT_ROOT / "demo_subscriptions.db"

# Start with a fresh demo database.
if database.DB_PATH.exists():
    database.DB_PATH.unlink()

database.initialize_database()

from agents import Runner, SQLiteSession, ToolCallItem
from agent import subscription_agent

memory_db = PROJECT_ROOT / "demo_memory.db"
if memory_db.exists():
    memory_db.unlink()

session = SQLiteSession("subscription_demo", db_path=memory_db)

print("✓ Demo ready")
print("✓ Database ready")
print("✓ Agent ready")

✓ Demo ready
✓ Database ready
✓ Agent ready


## Demo 1 — Add subscriptions

The agent uses the `add_subscription` tool to store two subscriptions.

In [2]:
result = await Runner.run(
    subscription_agent,
    "Add Netflix for ₹649 per month and Spotify for ₹1199 per month.",
    session=session,
)

print("USER:")
print("Add Netflix for ₹649 and Spotify for ₹1199.")
print("\nTOOL CALLS:")
for item in result.new_items:
    if isinstance(item, ToolCallItem):
        print("→", getattr(item, "tool_name", "tool"))

print("\nAGENT:")
print(result.final_output)

USER:
Add Netflix for ₹649 and Spotify for ₹1199.

TOOL CALLS:
→ add_subscription
→ add_subscription

AGENT:
✅ Netflix (₹649/month) and Spotify (₹1,199/month) have been added to your subscription list. Let me know if you’d like to check your total spending, set a budget, or do anything else!


## Demo 2 — Multi-step budget decision

We set a budget and ask what to cancel. The agent must obtain the budget, monthly total, and subscriptions before making its recommendation.

In [3]:
result = await Runner.run(
    subscription_agent,
    "Set my monthly subscription budget to ₹1500. Then tell me what subscription I should consider cancelling to get back under budget.",
    session=session,
)

print("USER:")
print("Set my budget to ₹1500 and tell me what I should cancel.")
print("\nTOOL CALLS / STEPS:")

step = 1
for item in result.new_items:
    if isinstance(item, ToolCallItem):
        print(f"Step {step}: {getattr(item, 'tool_name', 'tool')}")
        step += 1

print("\nAGENT:")
print(result.final_output)

USER:
Set my budget to ₹1500 and tell me what I should cancel.

TOOL CALLS / STEPS:
Step 1: set_monthly_budget
Step 2: get_monthly_total
Step 3: get_subscriptions

AGENT:
Your new monthly budget is **₹1,500**.

**Current spending**:  
- Netflix: ₹649/month  
- Spotify: ₹1,199/month  
- **Total**: ₹1,848/month  

You’re **₹348 over budget**.

### Recommendation to get back under budget

- **Cancel Netflix** (₹649/month).  
  - New total: ₹1,199/month.  
  - You’ll be **₹301 under** your ₹1,500 budget.  

Cancelling Netflix brings you comfortably below budget while keeping Spotify active. If you’d prefer to keep Netflix, we can explore other options or adjust the budget. Let me know what you’d like to do next!


### What this demonstrates

The stored subscriptions should total:

```text
Netflix  = ₹649
Spotify  = ₹1199
Total    = ₹1848
Budget   = ₹1500
```

The agent is over budget and should recommend a subscription to cancel. It does **not** automatically cancel anything.

## Demo 3 — Memory

We do not provide the subscription names or prices again. The agent can use the information stored during the earlier turn.

In [4]:
result = await Runner.run(
    subscription_agent,
    "How much will my current subscriptions cost per year?",
    session=session,
)

print("USER:")
print("How much will my current subscriptions cost per year?")
print("\nTOOL CALLS / STEPS:")

for item in result.new_items:
    if isinstance(item, ToolCallItem):
        print("→", getattr(item, "tool_name", "tool"))

print("\nAGENT:")
print(result.final_output)

USER:
How much will my current subscriptions cost per year?

TOOL CALLS / STEPS:
→ get_yearly_cost

AGENT:
Your current subscriptions would cost you **₹22,176 per year** (₹1,848 × 12).
